# How a Neural Network Learns XOR — Step by Step

## The 4 Steps of Learning

1. **Forward Pass**: Feed input through the network, get a prediction
2. **Calculate Error**: How far off was the prediction?
3. **Backward Pass**: Calculate which direction each weight should move (gradient)
4. **Update**: Nudge each weight a small step in that direction

---

## Network Structure

```
Input A ──w1──→ [H1] ──w5──→ [Output]
       ╲  ╱              ╱
        ╲╱              ╱
        ╱╲            ╱
       ╱  ╲          ╱
Input B ──w3──→ [H2] ──w6──→
       ──w2──→ [H1]
       ──w4──→ [H2]
```

- **w1**: Input A → Hidden 1
- **w2**: Input B → Hidden 1
- **w3**: Input A → Hidden 2
- **w4**: Input B → Hidden 2
- **w5**: Hidden 1 → Output
- **w6**: Hidden 2 → Output

In [ ]:
import numpy as np

# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    """Derivative of sigmoid, given the OUTPUT (not input)."""
    return output * (1 - output)

# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

print("XOR Truth Table:")
print("A  B  → Output")
for i in range(4):
    print(f"{X[i][0]}  {X[i][1]}  →   {y[i][0]}")

## Initial Weights (Random)

We start with small random values. Starting from all zeros would be a problem — all neurons would learn the same thing (symmetry problem).

In [ ]:
# Simple initial weights for easy-to-follow math
w1, w2 = 0.5, 0.3    # → H1
w3, w4 = 0.7, 0.2    # → H2
w5, w6 = 0.4, 0.6    # → Output
# Biases = 0 for simplicity

learning_rate = 0.5

print("Initial Weights:")
print(f"  w1 (A→H1) = {w1}")
print(f"  w2 (B→H1) = {w2}")
print(f"  w3 (A→H2) = {w3}")
print(f"  w4 (B→H2) = {w4}")
print(f"  w5 (H1→O) = {w5}")
print(f"  w6 (H2→O) = {w6}")
print(f"\nLearning rate = {learning_rate}")

---

## STEP 1: Forward Pass

Feed input `[1, 1]` through the network (expected output: **0**).

The forward pass is just multiplication and addition, then squish through sigmoid.

### Sigmoid Function

Sigmoid squishes any number into the range (0, 1):
- Large positive → close to 1
- Large negative → close to 0  
- Zero → exactly 0.5

In [ ]:
# Let's see what sigmoid does
print("Sigmoid examples:")
for val in [-5, -2, -1, 0, 1, 2, 5]:
    print(f"  sigmoid({val:+d}) = {sigmoid(val):.4f}")

print("\nKey insight: sigmoid(0) = 0.5 (the midpoint)")
print("It's like a soft switch: off(0) ←→ on(1)")

In [ ]:
# === FORWARD PASS with input [1, 1] ===
A, B = 1, 1
expected = 0

print("=" * 50)
print(f"FORWARD PASS: Input A={A}, B={B} (expected: {expected})")
print("=" * 50)

# Hidden layer: weighted sum
H1_input = (A * w1) + (B * w2)
H2_input = (A * w3) + (B * w4)
print(f"\n1. Hidden layer receives weighted sums:")
print(f"   H1_input = (A × w1) + (B × w2)")
print(f"           = ({A} × {w1}) + ({B} × {w2})")
print(f"           = {A*w1} + {B*w2}")
print(f"           = {H1_input}")
print(f"\n   H2_input = (A × w3) + (B × w4)")
print(f"           = ({A} × {w3}) + ({B} × {w4})")
print(f"           = {A*w3} + {B*w4}")
print(f"           = {H2_input}")

# Hidden layer: activation
H1_output = sigmoid(H1_input)
H2_output = sigmoid(H2_input)
print(f"\n2. Apply sigmoid activation:")
print(f"   H1_output = sigmoid({H1_input}) = {H1_output:.4f}")
print(f"   H2_output = sigmoid({H2_input}) = {H2_output:.4f}")

# Output layer
O_input = (H1_output * w5) + (H2_output * w6)
O_output = sigmoid(O_input)
print(f"\n3. Output layer:")
print(f"   O_input = (H1_output × w5) + (H2_output × w6)")
print(f"          = ({H1_output:.4f} × {w5}) + ({H2_output:.4f} × {w6})")
print(f"          = {H1_output*w5:.4f} + {H2_output*w6:.4f}")
print(f"          = {O_input:.4f}")
print(f"\n   O_output = sigmoid({O_input:.4f}) = {O_output:.4f}")

print(f"\n{'='*50}")
print(f"PREDICTION: {O_output:.4f}  (wanted: {expected})")
print(f"{'='*50}")

---

## STEP 2: Calculate Error

Simple subtraction. The sign tells us the direction:
- **Negative error** → prediction too high → need to push DOWN
- **Positive error** → prediction too low → need to push UP

In [ ]:
# === STEP 2: CALCULATE ERROR ===
error = expected - O_output

print("=" * 50)
print("STEP 2: CALCULATE ERROR")
print("=" * 50)
print(f"\n  error = expected - predicted")
print(f"        = {expected} - {O_output:.4f}")
print(f"        = {error:.4f}")
print(f"\n  Interpretation: error is NEGATIVE")
print(f"  → Our prediction ({O_output:.4f}) is TOO HIGH")
print(f"  → We need to push the output DOWN toward 0")

---

## STEP 3: Backward Pass (Backpropagation)

This is the "magic" step. We calculate **how much each weight contributed to the error**.

### The Chain Rule (intuition)

Think of it like blame assignment:
- The output neuron made the final mistake
- But it was fed by H1 and H2
- And H1/H2 were fed by the inputs through weights

We trace the error **backward** through the network, asking at each connection: "How much of this error is YOUR fault?"

### Sigmoid Derivative

The derivative tells us how "sensitive" the neuron is at its current output:
- Near 0.5 → very sensitive (steep slope) → big gradient
- Near 0 or 1 → barely sensitive (flat slope) → small gradient

```
sigmoid_derivative(output) = output × (1 - output)
```

In [ ]:
# Sigmoid derivative: shows sensitivity at different outputs
print("Sigmoid derivative at different outputs:")
print("(How 'moveable' is the neuron at this point?)\n")
for val in [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    deriv = sigmoid_derivative(val)
    bar = "█" * int(deriv * 80)
    print(f"  output={val:.2f} → derivative={deriv:.4f}  {bar}")

print("\n  Peak sensitivity at 0.5 (derivative = 0.25)")
print("  Near 0 or 1: almost no sensitivity (neuron is 'saturated')")

In [ ]:
# === BACKWARD PASS: Output Layer ===
print("=" * 50)
print("STEP 3a: Output Layer Gradient")
print("=" * 50)

# How sensitive is the output neuron?
O_sensitivity = sigmoid_derivative(O_output)
print(f"\n  Output sensitivity = sigmoid_derivative({O_output:.4f})")
print(f"                     = {O_output:.4f} × (1 - {O_output:.4f})")
print(f"                     = {O_output:.4f} × {1-O_output:.4f}")
print(f"                     = {O_sensitivity:.4f}")

# Delta for output: error × sensitivity
delta_output = error * O_sensitivity
print(f"\n  δ_output = error × sensitivity")
print(f"           = {error:.4f} × {O_sensitivity:.4f}")
print(f"           = {delta_output:.4f}")
print(f"\n  This means: 'The output should change by {delta_output:.4f}'")
print(f"  (negative = decrease the output)")

# How much should w5 and w6 change?
dw5 = delta_output * H1_output
dw6 = delta_output * H2_output
print(f"\n  How much blame does each weight get?")
print(f"  Δw5 = δ_output × H1_output = {delta_output:.4f} × {H1_output:.4f} = {dw5:.4f}")
print(f"  Δw6 = δ_output × H2_output = {delta_output:.4f} × {H2_output:.4f} = {dw6:.4f}")
print(f"\n  Logic: H1 was firing at {H1_output:.4f}, so w5 gets proportional blame")

In [ ]:
# === BACKWARD PASS: Hidden Layer ===
print("=" * 50)
print("STEP 3b: Hidden Layer Gradient")
print("=" * 50)
print("\nNow propagate error BACKWARD to hidden neurons:")

# How much error reached H1?
error_H1 = delta_output * w5
H1_sensitivity = sigmoid_derivative(H1_output)
delta_H1 = error_H1 * H1_sensitivity
print(f"\n  H1:")
print(f"    error_H1    = δ_output × w5 = {delta_output:.4f} × {w5} = {error_H1:.4f}")
print(f"    sensitivity = {H1_output:.4f} × (1 - {H1_output:.4f}) = {H1_sensitivity:.4f}")
print(f"    δ_H1        = {error_H1:.4f} × {H1_sensitivity:.4f} = {delta_H1:.4f}")

# How much error reached H2?
error_H2 = delta_output * w6
H2_sensitivity = sigmoid_derivative(H2_output)
delta_H2 = error_H2 * H2_sensitivity
print(f"\n  H2:")
print(f"    error_H2    = δ_output × w6 = {delta_output:.4f} × {w6} = {error_H2:.4f}")
print(f"    sensitivity = {H2_output:.4f} × (1 - {H2_output:.4f}) = {H2_sensitivity:.4f}")
print(f"    δ_H2        = {error_H2:.4f} × {H2_sensitivity:.4f} = {delta_H2:.4f}")

# Weight changes for input→hidden
dw1 = delta_H1 * A
dw2 = delta_H1 * B
dw3 = delta_H2 * A
dw4 = delta_H2 * B
print(f"\n  Weight changes (input × delta):")
print(f"    Δw1 = δ_H1 × A = {delta_H1:.4f} × {A} = {dw1:.4f}")
print(f"    Δw2 = δ_H1 × B = {delta_H1:.4f} × {B} = {dw2:.4f}")
print(f"    Δw3 = δ_H2 × A = {delta_H2:.4f} × {A} = {dw3:.4f}")
print(f"    Δw4 = δ_H2 × B = {delta_H2:.4f} × {B} = {dw4:.4f}")
print(f"\n  Note: if an input were 0, its weight gets Δ = 0 (no blame)")

---

## STEP 4: Update Weights

Apply the calculated changes. The learning rate (0.5) controls step size:
- Too big → overshoots, bounces around
- Too small → takes forever to learn
- Just right → steady convergence

```
new_weight = old_weight + learning_rate × Δweight
```

In [ ]:
# === STEP 4: UPDATE WEIGHTS ===
print("=" * 50)
print("STEP 4: UPDATE WEIGHTS")
print("=" * 50)
print(f"\nFormula: new = old + learning_rate × Δ")
print(f"Learning rate = {learning_rate}\n")

print(f"  {'Weight':<8} {'Old':>8} {'+ lr × Δ':>12} {'= New':>10}")
print(f"  {'─'*8} {'─'*8} {'─'*12} {'─'*10}")

weights_info = [
    ("w1", w1, dw1), ("w2", w2, dw2),
    ("w3", w3, dw3), ("w4", w4, dw4),
    ("w5", w5, dw5), ("w6", w6, dw6),
]

new_weights = {}
for name, old, delta in weights_info:
    change = learning_rate * delta
    new = old + change
    new_weights[name] = new
    direction = "↓" if change < 0 else "↑"
    print(f"  {name:<8} {old:>8.4f} {change:>+12.4f} {new:>10.4f}  {direction}")

print(f"\nAll weights DECREASED — because output was too high for [1,1]→0")
print(f"Each change was TINY but PURPOSEFUL (not random!)")

---

## Now Let's Watch Multiple Iterations

One pass barely changes anything. Let's run the full 4-step cycle on ALL training examples and watch the weights evolve over many epochs.

In [ ]:
# Full training loop with detailed logging at key moments

# Reset to initial weights (as matrices for efficiency)
np.random.seed(0)
w_ih = np.array([[0.5, 0.7], [0.3, 0.2]])   # input→hidden
w_ho = np.array([[0.4], [0.6]])               # hidden→output
b_h = np.zeros((1, 2))
b_o = np.zeros((1, 1))
lr = 0.5

log_epochs = [0, 1, 2, 3, 4, 10, 50, 100, 500, 1000, 5000, 9999]

print("Training XOR for 10,000 epochs...")
print("=" * 65)

for epoch in range(10000):
    # STEP 1: Forward pass (all 4 inputs at once)
    hidden = sigmoid(X @ w_ih + b_h)
    output = sigmoid(hidden @ w_ho + b_o)

    # STEP 2: Error
    error = y - output
    loss = np.mean(error ** 2)

    # STEP 3: Backward pass
    d_out = error * sigmoid_derivative(output)
    d_hid = (d_out @ w_ho.T) * sigmoid_derivative(hidden)

    # STEP 4: Update
    w_ho += hidden.T @ d_out * lr
    w_ih += X.T @ d_hid * lr
    b_o += np.sum(d_out, axis=0, keepdims=True) * lr
    b_h += np.sum(d_hid, axis=0, keepdims=True) * lr

    if epoch in log_epochs:
        pred = output.flatten()
        checks = ["✓" if (p < 0.1 and t == 0) or (p > 0.9 and t == 1) else "✗"
                   for p, t in zip(pred, y.flatten())]
        print(f"\nEpoch {epoch:>5d} | Loss: {loss:.6f}")
        print(f"  Predictions:  [0,0]→{pred[0]:.3f}{checks[0]}  "
              f"[0,1]→{pred[1]:.3f}{checks[1]}  "
              f"[1,0]→{pred[2]:.3f}{checks[2]}  "
              f"[1,1]→{pred[3]:.3f}{checks[3]}")
        print(f"  Weights:  w_ih={w_ih.flatten().round(3)}  w_ho={w_ho.flatten().round(3)}")

print(f"\n{'='*65}")
print(f"DONE! Final predictions vs expected:")
pred = output.flatten()
for i, (inp, exp) in enumerate(zip(X, y.flatten())):
    print(f"  {list(inp)} → {pred[i]:.4f}  (expected {exp})")

---

## Summary: What Just Happened

| Step | What | Math | Analogy |
|------|------|------|---------|
| **Forward** | Input → multiply by weights → sigmoid → output | `sigmoid(input × weight)` | Water flowing through pipes |
| **Error** | expected - predicted | `0 - 0.6688 = -0.6688` | "You're this far off" |
| **Backward** | Trace error back, calculate each weight's blame | `δ = error × derivative` | "It's YOUR fault this much" |
| **Update** | Nudge weights in the right direction | `w += lr × Δw` | Take one step downhill |

### Key Takeaways

1. **Weights start random** — not zero (symmetry breaking)
2. **Each update is calculated** — not random (gradient descent)
3. **Updates are tiny** — one step barely changes anything
4. **Thousands of steps** — repeated 4-step cycle across all training data
5. **The gradient (derivative)** — tells us the direction AND magnitude of change needed